# Capability 09 — Digital twins

Executed evidence over in-memory seed facts. Twins are computed, not persisted. Fraud and demand stay unknown.

In [1]:
import json
from datetime import datetime, timedelta
from decimal import Decimal
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from telco_digital.application.clock import FixedClock
from telco_digital.application.commands.commands import GetCustomerStateQuery
from telco_digital.application.queries.showcase import FactRecord, ProvenanceBlock, Retailer360
from telco_digital.application.seed import seed_demo_customers
from telco_digital.application.services.customer_state import get_customer_state
from telco_digital.infrastructure.memory import InMemoryUnitOfWork
from telco_digital.intelligence.behaviour import build_behaviour
from telco_digital.intelligence.churn import score_churn
from telco_digital.intelligence.digital_twin import assemble_customer_twin, assemble_retailer_twin
from telco_digital.intelligence.event_memory import EventMemoryService
from telco_digital.intelligence.event_memory.uow import UnitOfWorkEventMemoryQueries
from telco_digital.intelligence.features import CustomerFeatures, GraphFeatures
from telco_digital.intelligence.features.service import FeatureGroup
from telco_digital.intelligence.recommendations import PlanRepositoryCatalogue, build_recommendation

ROOT = Path(".")
for folder in ("outputs/tables", "outputs/plots", "artifacts"):
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
AS_OF = datetime.fromisoformat("2026-08-20T12:00:00+00:00")


async def features_from_uow(uow, customer_ref, as_of):
    customer = await uow.customers.get_by_ref(customer_ref)
    start_30 = as_of - timedelta(days=30)
    start_90 = as_of - timedelta(days=90)
    usage = [row for row in await uow.usage_events.list_as_of(customer.id, as_of) if row.occurred_at >= start_90]
    usage_30 = [row for row in usage if row.occurred_at >= start_30]
    recharges = [row for row in await uow.recharges.list_as_of(customer.id, as_of) if row.occurred_at >= start_90]
    recharge_30 = [row for row in recharges if row.occurred_at >= start_30]
    travels = list(await uow.travels.list_as_of(customer.id, as_of))
    service = [
        row
        for row in await uow.service_interactions.list_by_customer(customer.id)
        if row.occurred_at <= as_of and row.occurred_at >= start_90
    ]
    return CustomerFeatures(
        customer_id=customer.id,
        customer_ref=customer.customer_ref,
        as_of=as_of,
        computed_at=as_of,
        temporal={
            "usage": FeatureGroup(
                window_days=30,
                values={
                    "data_mb_30d": float(sum((row.data_mb for row in usage_30), Decimal("0"))),
                    "data_mb_90d": float(sum((row.data_mb for row in usage), Decimal("0"))),
                },
            ),
            "recharge": FeatureGroup(
                window_days=30,
                values={
                    "count_30d": len(recharge_30),
                    "amount_30d": float(sum((row.amount for row in recharge_30), Decimal("0"))),
                    "small_recharge_count_30d": sum(float(row.amount) <= 200 for row in recharge_30),
                    "frequent_small_recharge_evidence": sum(float(row.amount) <= 200 for row in recharge_30) >= 3,
                },
            ),
            "service": FeatureGroup(
                window_days=90,
                values={
                    "complaint_count_90d": sum(row.interaction_type == "COMPLAINT" for row in service),
                    "open_count": sum(row.status == "OPEN" for row in service),
                },
            ),
            "travel": FeatureGroup(
                window_days=365,
                values={"trip_count_365d": len(travels), "roaming_days_365d": 6 if travels else 0},
            ),
        },
        graph=GraphFeatures(available=False, values={}),
        provenance=("in-memory seed facts",),
        unknowns=("Neo4j graph features are unavailable; values are not assumed to be zero.",),
    )


async def customer_twin(uow, customer_ref, destination=None):
    observed = await get_customer_state(uow, GetCustomerStateQuery(customer_ref=customer_ref, as_of=AS_OF))
    context = await EventMemoryService(UnitOfWorkEventMemoryQueries(uow)).recall(
        customer_ref, AS_OF, destination=destination
    )
    features = await features_from_uow(uow, customer_ref, AS_OF)
    country = context.current_situation.destination if context.current_situation.destination_known else None
    catalogue = await PlanRepositoryCatalogue(uow.plans).list_roaming(country_code=country)
    return assemble_customer_twin(
        observed,
        features,
        context,
        build_behaviour(features, context.historical_episodes),
        score_churn(features),
        build_recommendation(context, catalogue),
    )


In [2]:
uow = InMemoryUnitOfWork()
await seed_demo_customers(uow, clock=FixedClock(AS_OF))
u001 = await customer_twin(uow, "U001", destination="SG")
u002 = await customer_twin(uow, "U002")
u004 = await customer_twin(uow, "U004")
assert u001.historical.top_plan == "ROAM_15"
assert u001.recommended.primary_plan_code == "ROAM_15"
assert {item.trait for item in u001.inferred.traits} >= {"FREQUENT_TRAVELLER", "HEAVY_DATA_USER"}
assert u004.predicted.churn_risk_band in {"MEDIUM", "HIGH"}
u001_row = pd.DataFrame(
    [
        {
            "section": "observed",
            "value": f"{u001.observed.country} / {u001.observed.current_plan_code}",
        },
        {
            "section": "historical",
            "value": f"{u001.historical.top_destination} {u001.historical.top_plan} {u001.historical.top_usage_gb}GB",
        },
        {
            "section": "inferred",
            "value": ", ".join(item.trait for item in u001.inferred.traits),
        },
        {
            "section": "predicted",
            "value": f"{u001.predicted.churn_risk_band} / fraud {u001.predicted.fraud_status}",
        },
        {
            "section": "recommended",
            "value": f"{u001.recommended.mode} {u001.recommended.primary_plan_code}",
        },
    ]
)
u001_row

,section,value
0,observed,LK / ROAM_15
1,historical,Singapore ROAM_15 11.4GB
2,inferred,"HEAVY_DATA_USER, FREQUENT_TRAVELLER, STREAMING..."
3,predicted,MEDIUM / fraud unknown
4,recommended,SCENARIO_BASED ROAM_15


In [3]:
as_of = AS_OF
provenance = ProvenanceBlock(source="live_database", as_of=as_of, dataset_version="poc-v1", table="sfa.sale")
retailer_facts = Retailer360(
    source="live_database",
    as_of=as_of,
    dataset_version="poc-v1",
    queried_at=as_of,
    retailer_ref="RET-001",
    name="Colombo Central",
    region="Western",
    status="ACTIVE",
    sales=(
        FactRecord(
            kind="sale",
            occurred_at=as_of,
            summary="Starter pack: 47 units, 18800",
            detail={"product_code": "SP-01", "quantity": 47, "amount": "18800"},
            provenance=provenance,
        ),
    ),
    inventory=(
        FactRecord(
            kind="inventory",
            occurred_at=as_of,
            summary="Starter pack: STOCK 18",
            detail={"product_code": "SP-01", "event_type": "STOCK", "quantity": 18},
            provenance=provenance,
        ),
    ),
)
retailer = assemble_retailer_twin(retailer_facts)
assert retailer.predicted.status == "unknown"
assert retailer.recommended.status == "unknown"
coverage = pd.DataFrame(
    [
        {"entity": "U001", "section": "observed", "populated": True},
        {"entity": "U001", "section": "recent", "populated": u001.recent.usage_mb_30d is not None},
        {"entity": "U001", "section": "historical", "populated": u001.historical.episode_count > 0},
        {"entity": "U001", "section": "relationships", "populated": u001.relationships.available},
        {"entity": "U001", "section": "inferred", "populated": bool(u001.inferred.traits)},
        {"entity": "U001", "section": "predicted_churn", "populated": u001.predicted.churn_risk_band is not None},
        {"entity": "U001", "section": "predicted_fraud", "populated": False},
        {"entity": "U001", "section": "recommended", "populated": u001.recommended.primary_plan_code is not None},
        {"entity": "RET-001", "section": "observed", "populated": retailer.observed.sale_count > 0},
        {"entity": "RET-001", "section": "historical", "populated": retailer.historical.total_quantity > 0},
        {"entity": "RET-001", "section": "predicted", "populated": False},
        {"entity": "RET-001", "section": "recommended", "populated": False},
    ]
)
retailer_row = pd.DataFrame(
    [
        {
            "retailer_ref": retailer.retailer_ref,
            "sale_count": retailer.observed.sale_count,
            "total_quantity": retailer.historical.total_quantity,
            "predicted": retailer.predicted.status,
            "recommended": retailer.recommended.status,
        }
    ]
)
u001_row.to_json(ROOT / "outputs" / "tables" / "u001_twin.json", orient="records", indent=2)
coverage.to_json(ROOT / "outputs" / "tables" / "section_coverage.json", orient="records", indent=2)
retailer_row.to_json(ROOT / "outputs" / "tables" / "retailer_twin.json", orient="records", indent=2)
metrics = {
    "u001_mode": u001.recommended.mode,
    "u001_primary": u001.recommended.primary_plan_code,
    "u001_traits": [item.trait for item in u001.inferred.traits],
    "u002_traits": [item.trait for item in u002.inferred.traits],
    "u004_churn_band": u004.predicted.churn_risk_band,
    "retailer_predicted": retailer.predicted.status,
    "unknowns": list(u001.unknowns[:3]),
}
(ROOT / "outputs" / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
coverage

,entity,section,populated
0,U001,observed,True
1,U001,recent,True
2,U001,historical,True
3,U001,relationships,False
4,U001,inferred,True
5,U001,predicted_churn,True
6,U001,predicted_fraud,False
7,U001,recommended,True
8,RET-001,observed,True
9,RET-001,historical,True


In [4]:
fig, axis = plt.subplots(figsize=(7, 3.4))
labels = coverage["section"] + " (" + coverage["entity"] + ")"
colors = ["#389e0d" if flag else "#bfbfbf" for flag in coverage["populated"]]
axis.barh(labels, coverage["populated"].astype(int), color=colors)
axis.set_title("Twin section coverage")
axis.set_xlabel("Populated")
fig.tight_layout()
fig.savefig(ROOT / "outputs" / "plots" / "section_coverage.png", dpi=120)
plt.close(fig)

fig, axis = plt.subplots(figsize=(5.5, 3.2))
axis.bar(
    ["U001 churn", "U004 churn", "U001 offer", "RET forecast"],
    [
        1 if u001.predicted.churn_risk_band else 0,
        1 if u004.predicted.churn_risk_band else 0,
        1 if u001.recommended.primary_plan_code else 0,
        0,
    ],
    color=["#1890ff", "#d48806", "#389e0d", "#bfbfbf"],
)
axis.set_title("Predicted vs recommended availability")
axis.set_ylim(0, 1.2)
fig.tight_layout()
fig.savefig(ROOT / "outputs" / "plots" / "predicted_vs_recommended.png", dpi=120)
plt.close(fig)
"plots written"

'plots written'